If the ingredient name contains "cream", "milk", etc. then say not lactose-free

If the ingredient name contains "meat", etc. not vegan/vegetarian

Contains "wheat", "flour", "pasta", etc. not gluten-free

In [1]:
import copy
import json
from collections import Counter

from data_supplementation_keywords import (
    GLUTEN_KEYWORDS,
    LACTOSE_KEYWORDS,
    NON_VEGAN_KEYWORDS,
    NON_VEGETARIAN_KEYWORDS,
)
from tqdm.auto import tqdm


In [2]:
with open("./structured_ingredients.json", "r") as f:
    structured_ingredients = json.load(f)

In [3]:
supplemented_structured_ingredients = []

In [4]:
def check_dietary_flag(name: str, keywords: list[str]) -> bool:
    """
    Returns False if any keyword is found in the ingredient name, else True (assumed compliancy)

    :param name: the ingredient name to check
    :type name: str
    :param keywords: the list of keywords to check for
    :type keywords: list[str]

    :return: False if any keyword is found, else True
    :rtype: bool
    """

    name_lower = name.lower()

    return not any(keyword in name_lower for keyword in keywords)

In [5]:
for ingredient in tqdm(structured_ingredients):
    supplemented_ingredient = copy.deepcopy(ingredient)

    name = ingredient["name"]
    nutritional_information = supplemented_ingredient["nutritional_information"]

    if nutritional_information["is_lactose_free"] is None:
        nutritional_information["is_lactose_free"] = check_dietary_flag(name, LACTOSE_KEYWORDS)

    if nutritional_information["is_gluten_free"] is None:
        nutritional_information["is_gluten_free"] = check_dietary_flag(name, GLUTEN_KEYWORDS)

    if nutritional_information["is_vegetarian"] is None:
        nutritional_information["is_vegetarian"] = check_dietary_flag(name, NON_VEGETARIAN_KEYWORDS)

    # if the ingredient is not vegetarian, it cannot be vegan. No need to check further
    if nutritional_information["is_vegetarian"] is False:
        nutritional_information["is_vegan"] = False
    elif nutritional_information["is_vegan"] is None:
        nutritional_information["is_vegan"] = check_dietary_flag(name, NON_VEGAN_KEYWORDS)

    supplemented_structured_ingredients.append(supplemented_ingredient)


  0%|          | 0/16076 [00:00<?, ?it/s]

In [6]:
# Get counts of all null values for nutritional information fields after supplementation

counter = Counter()

print(f"Total ingredients after supplementation: {len(supplemented_structured_ingredients)}")

print("Null value counts for nutritional information fields after supplementation:")
for field in supplemented_structured_ingredients[0]["nutritional_information"].keys():
    null_count = sum(
        1 for ingredient in supplemented_structured_ingredients if ingredient["nutritional_information"][field] is None
    )
    counter[field] = null_count
    print(f"- {field}: {null_count}")

Total ingredients after supplementation: 16076
Null value counts for nutritional information fields after supplementation:
- calories: 396
- carbohydrates: 427
- sugar: 2800
- protein: 326
- fat: 366
- saturated_fat: 2874
- fiber: 3026
- sodium: 460
- is_gluten_free: 0
- is_lactose_free: 0
- is_vegetarian: 0
- is_vegan: 0


In [7]:
json.dump(
    supplemented_structured_ingredients,
    open("./supplemented_structured_ingredients.json", "w"),
    indent=4,
)

In [8]:
import sys

sys.path.append("..")

from models import Ingredient, Recipe, NutritionalInformation, DietaryTag

In [9]:
ingredient_lookup: dict[str, Ingredient] = {}

for ingredient_data in supplemented_structured_ingredients:
    ingredient_lookup[ingredient_data["name"]] = Ingredient(
        name=ingredient_data["name"],
        nutritional_information=NutritionalInformation(**ingredient_data["nutritional_information"]),
    )

In [10]:
with open("./structured_recipes.json", "r") as f:
    structured_recipes_json = json.load(f)

tag_map = {tag.name: tag for tag in DietaryTag}

for recipe_data in tqdm(structured_recipes_json, desc="Computing Recipe Nutritional Information"):
    recipe = Recipe(
        name=recipe_data["name"],
        ingredients={entry["ingredient"]: entry["quantity"] for entry in recipe_data["ingredients"]},
        dietary_tags=[tag_map[tag] for tag in recipe_data["dietary_tags"] if tag in tag_map],
        instructions=recipe_data.get("instructions", []),
    )

    nutritional_information = recipe.compute_nutritional_information(ingredient_lookup)
    recipe_data["nutritional_information"] = nutritional_information.to_dict()

    # also update the dietary tags for the recipe
    new_dietary_tags = []

    if nutritional_information.is_lactose_free:
        new_dietary_tags.append(DietaryTag.LACTOSE_FREE.name)
    if nutritional_information.is_gluten_free:
        new_dietary_tags.append(DietaryTag.GLUTEN_FREE.name)
    if nutritional_information.is_vegetarian:
        new_dietary_tags.append(DietaryTag.VEGETARIAN.name)
    if nutritional_information.is_vegan:
        new_dietary_tags.append(DietaryTag.VEGAN.name)

    recipe_data["dietary_tags"] = new_dietary_tags

Computing Recipe Nutritional Information:   0%|          | 0/19716 [00:00<?, ?it/s]

In [11]:
json.dump(
    structured_recipes_json,
    open("./supplemented_structured_recipes.json", "w"),
    indent=4,
)